In [2]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json


In [3]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [4]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [6]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [7]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [8]:
@F.udf(returnType="STRING")
def get_properties(url):
   json_request = requests.get(url).json()
   return json.dumps(json_request["result"]["properties"])

In [9]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))
    
bronze_instance = StarWarsBronze(spark, **options)


In [10]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [11]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-01-11 13:49:...|        Cliegg Lars| 62|https://www.swapi...|{"height": "183",...|
|2025-01-11 13:49:...|  Poggle the Lesser| 63|https://www.swapi...|{"height": "183",...|
|2025-01-11 13:49:...|    Luminara Unduli| 64|https://www.swapi...|{"height": "170",...|
|2025-01-11 13:49:...|      Barriss Offee| 65|https://www.swapi...|{"height": "166",...|
|2025-01-11 13:49:...|              Dormé| 66|https://www.swapi...|{"height": "165",...|
|2025-01-11 13:49:...|              Dooku| 67|https://www.swapi...|{"height": "193",...|
|2025-01-11 13:49:...|Bail Prestor Organa| 68|https://www.swapi...|{"height": "191",...|
|2025-01-11 13:49:...|         Jango Fett| 69|https://www.swapi...|{"height": "183",...|
|2025-01

In [12]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [13]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [14]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [15]:
#with load filter and transformation
class StarWarsSilver(silver.Silver):
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("uid <= '25'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)

In [16]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute("people", "planets")

In [17]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+--------------------------+---------------------+---+------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 18
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-01-11 13:50:

In [19]:
#without load filter 
silver_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [20]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-11 13:50:...|2025-01-11 13:49:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-11 13:50:...|2025-01-11 13:49:...|               C-3PO|  2|https://www.swapi...|{167, 75, n/a, go...|
|2025-01-11 13:50:...|2025-01-11 13:49:...|               R2-D2|  3|https://www.swapi...|{96, 32, n/a, whi...|
|2025-01-11 13:50:...|2025-01-11 13:49:...|         Darth Vader|  4|https://www.swapi...|{202, 136, none, ...|
|2025-01-11 13:50:...|2025-01-11 13:49:...|         Leia Organa|  5|https://www.swapi...|{150, 49, brown, ...|
|2025-01-11 13:50:...|2025-01-11 13:49:...|           Owen Lars|  6|https://www.swapi...|{178, 120,

In [21]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 60
+--------------------+--------------------+-----------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|       name|uid|                 url|          properties|
+--------------------+--------------------+-----------+---+--------------------+--------------------+
|2025-01-11 13:51:...|2025-01-11 13:50:...|   Mon Cala| 31|https://www.swapi...|{11030, 21, 398, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|  Chandrila| 32|https://www.swapi...|{13500, 20, 368, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|    Sullust| 33|https://www.swapi...|{12780, 20, 263, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|   Toydaria| 34|https://www.swapi...|{7900, 21, 184, 1...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|  Malastare| 35|https://www.swapi...|{18880, 26, 201, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|   Dathomir| 36|https://www.swapi...|{10480, 24, 491, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|     Ryloth| 37|https://ww

# 3 Replace Where

In [22]:
class StarWarsSilver(silver.Silver): 
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("uid > '0'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
    def get_replace_condition(self, sdf: DataFrame, table: str) -> str:
        return "uid > 0"
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load(filter="custom").transform().write(mode="replace").execute("people", "planets")

In [23]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|2025-01-11 13:51:...|2025-01-11 13:49:...|        Cliegg Lars| 62|https://www.swapi...|{183, unknown, br...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|  Poggle the Lesser| 63|https://www.swapi...|{183, 80, none, g...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|    Luminara Unduli| 64|https://www.swapi...|{170, 56.2, black...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Barriss Offee| 65|https://www.swapi...|{166, 50, black, ...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|              Dormé| 66|https://www.swapi...|{165, unknown, br...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|              Dooku| 67|https://www.swapi...|{193, 80, white, ..

# 4 Append

In [24]:
class StarWarsSilver(silver.Silver): 
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("custom == 'custom'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write().execute("people", "planets") #default mode is append

In [25]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|2025-01-11 13:51:...|2025-01-11 13:49:...|        Cliegg Lars| 62|https://www.swapi...|{183, unknown, br...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|  Poggle the Lesser| 63|https://www.swapi...|{183, 80, none, g...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|    Luminara Unduli| 64|https://www.swapi...|{170, 56.2, black...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Barriss Offee| 65|https://www.swapi...|{166, 50, black, ...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|              Dormé| 66|https://www.swapi...|{165, unknown, br...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|              Dooku| 67|https://www.swapi...|{193, 80, white, .

# 5 Merge

In [26]:
class StarWarsSilver(silver.Silver): 
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
      
    def get_delta_merge_builder(self, sdf: DataFrame, delta_table: DeltaTable) -> DeltaMergeBuilder:
        merge_condition = "target.url = source.url" 
        builder = delta_table.alias("target").merge(sdf.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write(mode="merge").execute("people", "planets")

In [27]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84, blond, ...|
|2025-01-11 13:51:...|2025-01-11 13:49:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84,

In [28]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 120
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-01-11 13:51:...|2025-01-11 13:50:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-01-11 13:51:...|2025-01-11 13:50:..

# 6 Clean Up

In [29]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]